## Parameters

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("model_catalog", "")
dbutils.widgets.text("model_schema", "")
dbutils.widgets.text("model_name", "")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
table_name = dbutils.widgets.get("table_name").strip()
model_catalog = dbutils.widgets.get("model_catalog").strip()
model_schema = dbutils.widgets.get("model_schema").strip()
model_name = dbutils.widgets.get("model_name").strip()

params = {
    "catalog": catalog,
    "schema": schema,
    "table_name": table_name,
    "model_catalog": model_catalog,
    "model_schema": model_schema,
    "model_name": model_name,
}
if not all(params.values()):
    missing = ", ".join(name for name, value in params.items() if not value)
    raise ValueError(f"All parameters must be non-empty; missing: {missing}")

full_table = f"{catalog}.{schema}.{table_name}"
full_model_name = f"{model_catalog}.{model_schema}.{model_name}"

## Load data

In [ ]:
pdf = spark.table(full_table).toPandas()

y = pdf["label"]
X = pdf[[c for c in pdf.columns if c.startswith("feature_")]]

print(f"Loaded {len(pdf)} rows with {X.shape[1]} features from {full_table}")

## Train & convert to ONNX

In [ ]:
import xgboost as xgb
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

model = xgb.XGBClassifier(random_state=42, eval_metric="logloss")
model.fit(X.to_numpy(), y)

initial_type = [("float_input", FloatTensorType([None, X.shape[1]]))]
onnx_model = convert_xgboost(model, initial_types=initial_type)

print(f"Trained XGBClassifier on {len(X)} rows and converted to ONNX")

## Register in UC Model Registry

In [ ]:
import tempfile
from pathlib import Path

import mlflow
import mlflow.pyfunc
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from mlflow.models import ModelSignature
from mlflow.types.schema import ColSpec, Schema

mlflow.set_registry_uri("databricks-uc")


class XGBoostOnnxPyFunc(mlflow.pyfunc.PythonModel):
    def __init__(self, registered_model_name, feature_names):
        self.registered_model_name = registered_model_name
        self.feature_names = feature_names

    def load_context(self, context):
        self.session = ort.InferenceSession(context.artifacts["onnx_model"])
        self.input_name = self.session.get_inputs()[0].name
        self.proba_output_name = next(
            output.name for output in self.session.get_outputs() if len(output.shape) == 2
        )

    def predict(self, context, model_input):
        features = model_input.loc[:, self.feature_names].to_numpy(dtype=np.float32)
        proba = self.session.run(
            [self.proba_output_name],
            {self.input_name: features},
        )[0]
        positive_proba = proba[:, 1].astype(np.float64)
        return pd.DataFrame(
            {
                "target": (positive_proba >= 0.5).astype(np.int64),
                "proba": positive_proba,
                "model_name": self.registered_model_name,
            }
        )


input_schema = Schema([ColSpec("double", column) for column in X.columns])
output_schema = Schema(
    [
        ColSpec("long", "target"),
        ColSpec("double", "proba"),
        ColSpec("string", "model_name"),
    ]
)
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with tempfile.TemporaryDirectory() as tmpdir:
    onnx_model_path = Path(tmpdir) / "model.onnx"
    onnx.save_model(onnx_model, str(onnx_model_path))

    with mlflow.start_run():
        model_info = mlflow.pyfunc.log_model(
            name="model",
            python_model=XGBoostOnnxPyFunc(
                registered_model_name=model_name,
                feature_names=list(X.columns),
            ),
            artifacts={"onnx_model": str(onnx_model_path)},
            signature=signature,
            pip_requirements=[
                "mlflow",
                "onnxruntime",
                "numpy",
                "pandas",
                "cloudpickle",
            ],
        )
        loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
        sample_predictions = loaded_model.predict(X.head(2))
        assert list(sample_predictions.columns) == ["target", "proba", "model_name"]
        assert len(sample_predictions) == 2
        assert sample_predictions["model_name"].tolist() == [model_name, model_name]

        registered = mlflow.register_model(model_info.model_uri, full_model_name)

print(f"Registered {registered.name} version {registered.version}")